
# Week 01 Assignment  
## Data Quality, Evaluation, Scaling, and Encoding

**Student name: Shihab Shariar**   

This is a small assignment that connects topics from Module 1, 2, and 3.  
You must complete it in this Colab notebook.

You will need to use concepts that appeared in the videos:
- Module 1 and 2: basic descriptive statistics, proportions, confusion matrix, accuracy, precision, recall
- Module 3: standardization, min max scaling, nominal vs ordinal, one hot encoding, ordinal encoding, Euclidean and Manhattan distance

Please do not use any extra libraries beyond `pandas`, `numpy`.



---
## 0. Setup and Dataset

We will use a dataset that should have columns given below:

- `user_id`  
- `age`  
- `monthly_income` (numeric)  
- `daily_screen_time_min` (numeric)  
- `daily_app_opens` (numeric)  
- `true_label` and `pred_label` for a binary classification task (0 or 1)  
- `satisfaction_level` (for example: `Low`, `Medium`, `High`)  
- `city_type` (for example: `Urban`, `Suburban`, `Rural`)


In [51]:
# Cell 1: Imports
import pandas as pd
import numpy as np

In [52]:
# Cell 2: Load the dataset (Already done for you)
df = pd.read_csv("https://drive.google.com/uc?export=download&id=1OmDDCh4MD1TtvAemnwVDyz5zwCIXJ220")

# Show first few rows
df.head()

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type
0,1,43,3734.19,109,48,0,0,Medium,Suburban
1,2,49,2594.19,194,7,0,0,Low,Urban
2,3,19,3550.47,146,36,1,0,High,Rural
3,4,19,3821.18,287,14,1,0,High,Suburban
4,5,63,1750.84,66,46,0,0,Medium,Suburban



### 0.1 Check your dataset

1. Confirm that the dataset loaded correctly.  
2. Check that you have at least these columns:  
   - numeric: `age`, `monthly_income`, `daily_screen_time_min`, `daily_app_opens`  
   - labels: `true_label`, `pred_label`  
   - categorical: `satisfaction_level`, `city_type`  



---
## Part A - Module 1 and 2 Review

In this part you will do simple descriptive statistics and basic classification evaluation.



### Q1. Descriptive statistics on a numeric feature

Choose one numeric column, for example `daily_screen_time_min`.


In [53]:
# Q1.1: Choose your numeric column here [We already write this ans]
num_col = "monthly_income"

df[num_col].describe()

,monthly_income
count,100.000000
mean,2885.745000
std,898.124693
min,1000.000000
25%,2379.165000
50%,2894.020000
75%,3593.165000
max,5049.400000



> **Q1.2 Short answer: [Marks: 05]**  
> Look at the count, mean, min, max, and standard deviation for your chosen column.  
> In 2 to 3 sentences, comment on what you see.  
> For example, does the max look very far from the mean, or does it look quite close?

Write your answer here:

>  The monthly income column has 100 records, which is why the count is showing 100.
>  The average income is approximately 2885.74 and the standard deviation 898.12 shows that the incomes vary moderately. Since the mean and median are almost the same, the data appears to be fairly balance without any major outliners, The minimum income is 1000 and the maximum income is 5049.40. Although the maximum income is higher than the average but not by a huge margin, means income values are evenly distributed.



### Q2. Proportion of positive class

Use the `true_label` column, where 1 means "positive" and 0 means "negative".


In [54]:
# Q2.1: Compute proportion of positive class [We already write this ans]
label_col = "true_label"

positive_count = (df[label_col] == 1).sum()
total_count = df.shape[0]
positive_proportion = positive_count / total_count

print("Positive count:", positive_count)
print("Total samples:", total_count)
print("Proportion of positive class:", positive_proportion)

Positive count: 52
Total samples: 100
Proportion of positive class: 0.52



> **Q2.2 Short answer: [5 marks]**  
> In 1 to 2 sentences, explain what this proportion tells you about your dataset.  
> For example, is the dataset balanced between 0 and 1, or is one class much more common?

Write your answer here:

> The dataset contains 52 positive samples and 48 negative samples. The positive class proportion is 0.52 which means that both classes are alomost equally represented. So we can say the dataset is fairly balanced and should not cause any significant class imbalance issues.   
>  
>  



### Q3. Confusion matrix and basic metrics

For this question, use:
- `true_label` as the actual label  
- `pred_label` as the model prediction


In [55]:
# Q3.1: Manually compute TP, TN, FP, FN [We already write this ans]
true_col = "true_label"
pred_col = "pred_label"

tp = ((df[true_col] == 1) & (df[pred_col] == 1)).sum()
tn = ((df[true_col] == 0) & (df[pred_col] == 0)).sum()
fp = ((df[true_col] == 0) & (df[pred_col] == 1)).sum()
fn = ((df[true_col] == 1) & (df[pred_col] == 0)).sum()

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

TP: 28
TN: 27
FP: 21
FN: 24


In [56]:
# Q3.2: Compute accuracy, precision, recall [We already write this ans]
accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.55
Precision: 0.5714285714285714
Recall: 0.5384615384615384



> **Q3.3 Short answer: [10 marks]**  
> In 3 to 4 sentences, briefly comment on the model using these three metrics.  
> For example, is the model catching most positives (high recall) or being careful when it predicts positive (high precision)?

Write your answer here:

>  The accuracy of 55% means that the classification was correct for more than half of the samples. The precision of 57.14% means that the positive class predictions were correct a little over half of the time.
>  The recall of 53.85% shows that the classification performance is moderate because the the model predicted just over half of the actual positive cases.



---
## Part B - Module 3: Scaling and Encoding

Now we will pick a few features and apply scaling and encoding.



### Q4. Standardization and Min max scaling

Use one numeric column, `monthly_income`.


In [57]:
# Q4.1: Choose the numeric column [2 marks]
mi = df["monthly_income"]

In [58]:
# Q4.2: Standardization with z-score [10 marks]
mi_mean = mi.mean()
mi_std = mi.std()
mi_std_scaled = (mi - mi_mean) / mi_std
print(f"scaled value minimum : {mi_std_scaled.min()}")
print(f"scaled value maximum : {mi_std_scaled.max()}")
mi_std_scaled

scaled value minimum : -2.0996472034707563
scaled value maximum : 2.4090808513481488


,monthly_income
0,0.944685
1,-0.324626
2,0.740126
3,1.041542
4,-1.263639
...,...
95,1.213813
96,0.978734
97,-0.611613
98,0.135599


In [59]:
# Q4.3: Min max scaling implementation [10 marks]
mi_min = mi.min()
mi_max = mi.max()
mi_min_max_scaled = (mi - mi_min) / (mi_max - mi_min)
print(f"scaled value minimum : {mi_min_max_scaled.min()}")
print(f"scaled value maximum : {mi_min_max_scaled.max()}")
mi_min_max_scaled



scaled value minimum : 0.0
scaled value maximum : 1.0


,monthly_income
0,0.675209
1,0.393685
2,0.629839
3,0.696691
4,0.185420
...,...
95,0.734899
96,0.682760
97,0.330034
98,0.495760



> **Q4.4 Short answer: [3 marks]**  
> Compare the standardized and min max scaled columns in 2 to 3 sentences.  
> Mention what kind of range each one uses and how the numbers look.

Write your answer here:

>  The Min-Max scaled column transforms the values into a fixed range between 0 and 1, so all values are positive and easy to compare.
>  The standardized column does not have a fixed range. Its values are centered around 0 with some values being positive and others negative depending on whether they are below or above the mean.
>


### Q5. One hot and ordinal encoding

We will use:
- `city_type` as a nominal feature  
- `satisfaction_level` as an ordinal feature with order `Low` < `Medium` < `High`  


In [60]:
# Q5.1: One hot encoding for city_type using pandas [10 marks]
city_type_one_hot = pd.get_dummies(df["city_type"], prefix="encoded", dtype=np.int64)
city_type_one_hot

,encoded_Rural,encoded_Suburban,encoded_Urban
0,0,1,0
1,0,0,1
2,1,0,0
3,0,1,0
4,0,1,0
...,...,...,...
95,0,0,1
96,0,0,1
97,1,0,0
98,0,0,1


In [61]:
# Q5.2: Attach one hot encoded columns to df [5 marks]
df = pd.concat([df, city_type_one_hot], axis=1)
df

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type,encoded_Rural,encoded_Suburban,encoded_Urban
0,1,43,3734.19,109,48,0,0,Medium,Suburban,0,1,0
1,2,49,2594.19,194,7,0,0,Low,Urban,0,0,1
2,3,19,3550.47,146,36,1,0,High,Rural,1,0,0
3,4,19,3821.18,287,14,1,0,High,Suburban,0,1,0
4,5,63,1750.84,66,46,0,0,Medium,Suburban,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,20,3975.90,259,15,0,1,Medium,Urban,0,0,1
96,97,52,3764.77,295,16,1,0,Low,Urban,0,0,1
97,98,35,2336.44,108,46,0,1,High,Rural,1,0,0
98,99,18,3007.53,202,42,1,1,High,Urban,0,0,1


In [62]:
# Q5.3: Ordinal encoding for satisfaction_level [10 marks]
satisfaction_level_mapping = {"Low": 1, "Medium": 2, "High": 3}
df["satisfaction_level_encoded"] = df["satisfaction_level"].map(satisfaction_level_mapping)
df


,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type,encoded_Rural,encoded_Suburban,encoded_Urban,satisfaction_level_encoded
0,1,43,3734.19,109,48,0,0,Medium,Suburban,0,1,0,2
1,2,49,2594.19,194,7,0,0,Low,Urban,0,0,1,1
2,3,19,3550.47,146,36,1,0,High,Rural,1,0,0,3
3,4,19,3821.18,287,14,1,0,High,Suburban,0,1,0,3
4,5,63,1750.84,66,46,0,0,Medium,Suburban,0,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,20,3975.90,259,15,0,1,Medium,Urban,0,0,1,2
96,97,52,3764.77,295,16,1,0,Low,Urban,0,0,1,1
97,98,35,2336.44,108,46,0,1,High,Rural,1,0,0,3
98,99,18,3007.53,202,42,1,1,High,Urban,0,0,1,3



> **Q5.4 Short answer: [5 marks]**  
> In 2 to 3 sentences, explain why one hot encoding is suitable for `city_type`  
> and why ordinal encoding is suitable for `satisfaction_level`.

Write your answer here:

>  One hot encoding is suitable for city_type because the categories like rural, suburban and urban does not have any natural order, it converts each category into separate binary column without implying that one category is greater or smaller than another.
>  Ordinal encoding is suitable for satisfactory_level because the categories have a meaningful order like low, high and medium. This allowing the model to preserve their ranking.
>  



---
## Part C - Module 3: Distances between users

For this small part we will work with vectors based on scaled numeric features.



### Q6. Euclidean and Manhattan distance

Build 2D vectors for user 0 and user 1 using:
- `income_std`  
- `daily_app_opens` (or its min max scaled version if you prefer)


In [69]:
# Q6.1: Build 2D vectors for first two users [We already write this ans]
vec_cols = ["monthly_income", "daily_app_opens"]

v1 = df.loc[0, vec_cols].values
v2 = df.loc[1, vec_cols].values

print("v1:", v1)
print("v2:", v2)

v1: [np.float64(3734.19) np.int64(48)]
v2: [np.float64(2594.19) np.int64(7)]


In [77]:
# Q6.2: Euclidean distance computation [5 marks]
eu_dst = np.linalg.norm(v1-v2)
print(eu_dst.round(2))

1140.74


In [75]:
# Q6.3: Manhattan distance computation [5 marks]
man_dst = np.linalg.norm( v1-v2, ord=1)
print(man_dst)

1181.0



> **Q6.4 Short answer: [5 marks]**  
> Which one is larger in your result, Euclidean or Manhattan distance  
> and why does that usually happen based on their formulas?

Write your answer here:

>  Here the Manhattan distance 1181 is larger than the Euclidean distance 1140.74. This happens because Manhattan distance adds the absolute differences of each feature while Euclidean distance calculates the shortest straight line distance between the two points.
>  
>  



---
## Final Reflection [10 marks]

> In 4 to 6 sentences, describe how the three modules connect in this assignment.  
> Mention:
> - One idea from Module 1 or 2 that you used  
> - One idea from Module 3 that you used  
> - How these ideas together help you understand a dataset more deeply

Write your reflection here:

> From module 1 and 2 understanding the descriptive statistics like mean, median and standard deviation and basic probability cencepts helped me to analyze the data distribution and behaviour. From module 3 implementing techniques like features scaling, encoding and distance metrics like euclidean and manhattan distances helped me to answer the questions.
Combining these concepts help me gain a deeper and accurate understanding of dataset's structure before feeding it into machine learning models.  



## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।
